# 🚇 MTA Subway Schedule (Static GTFS) - Data Ingest
### Powered by Elasticsearch Serverless + Mistral

This notebook ingests the MTA's **static subway schedule** (GTFS) into Elasticsearch: every station (with geo location), every route, and - optionally - the timetable.

**What this notebook does:**
1. Downloads the MTA subway GTFS zip from [MTA Developers](https://www.mta.info/developers) - no API key
2. Parses `stops.txt` and `routes.txt`
3. Bulk-indexes stations (as `geo_point`s) and routes into Elasticsearch
4. **(Optional)** ingests a slice of the timetable (`trips` + `stop_times`)

Pairs nicely with the **real-time** feed notebook: schedule tells you when a train *should* come, real-time tells you when it *will*.

---
> **Data source:** MTA subway static GTFS, `https://rrgtfsfeeds.s3.amazonaws.com/gtfs_subway.zip` (public).

## 0. Install dependencies

In [ ]:
!pip install elasticsearch requests --quiet

In [ ]:
import requests, io, zipfile, csv
from elasticsearch import Elasticsearch, helpers

print('✅ Imports ready')

## 1. Connect to Elastic Serverless
> **Elastic Cloud Console → Your Project → Connection Details**

In [ ]:
# 🔑 Fill these in
ELASTIC_ENDPOINT = 'https://your-project.es.region.aws.elastic.cloud'
ELASTIC_API_KEY  = 'your-elastic-api-key'

es = Elasticsearch(ELASTIC_ENDPOINT, api_key=ELASTIC_API_KEY)
print(f'✅ Connected to Elasticsearch {es.info()["version"]["number"]}')

## 2. Download and open the GTFS zip

In [ ]:
GTFS_URL = 'https://rrgtfsfeeds.s3.amazonaws.com/gtfs_subway.zip'
resp = requests.get(GTFS_URL, timeout=120)
resp.raise_for_status()
zf = zipfile.ZipFile(io.BytesIO(resp.content))
print('✅ Downloaded GTFS. Files inside:')
print('  ', zf.namelist())

def read_csv(name):
    with zf.open(name) as f:
        return list(csv.DictReader(io.TextIOWrapper(f, 'utf-8-sig')))

stops  = read_csv('stops.txt')
routes = read_csv('routes.txt')
print(f'\n{len(stops)} stops, {len(routes)} routes')

## 3. Index the stations (with geo_point)

In [ ]:
STOPS_INDEX = 'mta_subway_stops'
if es.indices.exists(index=STOPS_INDEX):
    es.indices.delete(index=STOPS_INDEX)
es.indices.create(index=STOPS_INDEX, body={'mappings': {'properties': {
    'stop_id':   {'type': 'keyword'},
    'stop_name': {'type': 'text', 'fields': {'keyword': {'type': 'keyword'}}},
    'location':  {'type': 'geo_point'},
    'parent_station': {'type': 'keyword'},
}}})

def stop_doc(s):
    doc = {'stop_id': s['stop_id'], 'stop_name': s['stop_name'],
           'parent_station': s.get('parent_station') or None}
    try:
        doc['location'] = {'lat': float(s['stop_lat']), 'lon': float(s['stop_lon'])}
    except (ValueError, KeyError):
        pass
    return {k: v for k, v in doc.items() if v not in (None, '')}

helpers.bulk(es, [{'_index': STOPS_INDEX, '_source': stop_doc(s)} for s in stops], raise_on_error=False)
es.indices.refresh(index=STOPS_INDEX)
print(f'✅ Indexed {len(stops)} stops into {STOPS_INDEX}')

## 4. Index the routes

In [ ]:
ROUTES_INDEX = 'mta_subway_routes'
if es.indices.exists(index=ROUTES_INDEX):
    es.indices.delete(index=ROUTES_INDEX)
es.indices.create(index=ROUTES_INDEX, body={'mappings': {'properties': {
    'route_id':         {'type': 'keyword'},
    'route_short_name': {'type': 'keyword'},
    'route_long_name':  {'type': 'text'},
    'route_color':      {'type': 'keyword'},
    'route_desc':       {'type': 'text'},
}}})

def route_doc(r):
    doc = {'route_id': r.get('route_id'), 'route_short_name': r.get('route_short_name'),
           'route_long_name': r.get('route_long_name'), 'route_color': r.get('route_color'),
           'route_desc': r.get('route_desc')}
    return {k: v for k, v in doc.items() if v not in (None, '')}

helpers.bulk(es, [{'_index': ROUTES_INDEX, '_source': route_doc(r)} for r in routes], raise_on_error=False)
es.indices.refresh(index=ROUTES_INDEX)
print(f'✅ Indexed {len(routes)} routes into {ROUTES_INDEX}')

## 5. Verify

In [ ]:
print('Routes:')
for r in routes[:12]:
    print(f"  {r.get('route_short_name','?'):4s} {r.get('route_long_name','')}")

# Find subway stops near Times Square (geo query proves the geo_point works)
resp = es.search(index=STOPS_INDEX, size=5, body={
    'query': {'geo_distance': {'distance': '500m', 'location': {'lat': 40.7558, 'lon': -73.9870}}}
})
print('\nStops within 500m of Times Square:')
for h in resp['hits']['hits']:
    print('  •', h['_source']['stop_name'])

## 6. (Optional) Ingest a slice of the timetable
`stop_times.txt` is large (hundreds of thousands of rows). Ingest the scheduled stops for a single route so your agent can answer "when does the L run?" without loading everything.

In [ ]:
trips = read_csv('trips.txt')
ROUTE = 'L'  # change to any route_id

trip_ids = {t['trip_id'] for t in trips if t.get('route_id') == ROUTE}
print(f'{len(trip_ids)} scheduled {ROUTE} trips')

stop_times = read_csv('stop_times.txt')
sel = [st for st in stop_times if st['trip_id'] in trip_ids]
print(f'{len(sel)} stop-time rows for route {ROUTE}')

ST_INDEX = f'mta_schedule_{ROUTE.lower()}'
if es.indices.exists(index=ST_INDEX):
    es.indices.delete(index=ST_INDEX)
es.indices.create(index=ST_INDEX, body={'mappings': {'properties': {
    'trip_id': {'type': 'keyword'}, 'stop_id': {'type': 'keyword'},
    'arrival_time': {'type': 'keyword'}, 'departure_time': {'type': 'keyword'},
    'stop_sequence': {'type': 'integer'}, 'route_id': {'type': 'keyword'},
}}})
def st_doc(st):
    return {'trip_id': st['trip_id'], 'stop_id': st['stop_id'],
            'arrival_time': st.get('arrival_time'), 'departure_time': st.get('departure_time'),
            'stop_sequence': int(st['stop_sequence']) if st.get('stop_sequence') else None,
            'route_id': ROUTE}
helpers.bulk(es, [{'_index': ST_INDEX, '_source': st_doc(s)} for s in sel], raise_on_error=False)
es.indices.refresh(index=ST_INDEX)
print(f'✅ Indexed {len(sel)} scheduled stops for route {ROUTE} into {ST_INDEX}')

## 7. Going further with Mistral

- **Trip-planner agent:** give a Mistral agent function tools over `mta_subway_stops` (with geo) and `mta_subway_routes`, so it can answer "What's the closest station to the Met, and which lines stop there?"
- **Structured outputs:** have a Mistral model turn a free-text request ("get me from Astoria to Union Square") into structured JSON (origin, destination, lines) that you run against Elasticsearch.
- **Schedule vs. reality:** pair this with the real-time notebook and let the agent compare scheduled and live arrivals.

See [mistral_guide.md](mistral_guide.md) for calling Mistral directly.

---
## ✅ Indexes loaded
`mta_subway_stops` and `mta_subway_routes` are ready. Combine with the **real-time** feed (`mta_subway_realtime.ipynb`) to pair the scheduled and live pictures of the system.